# Week 3 — Watching Attention Work (English-only candidate)

**LLMs & You · Hampden-Sydney College · Fall 2026**

On Tuesday we read the paper *Attention is All You Need*. Today you open a
transformer it describes and look at the attention weights themselves.

This is the **English-only version** of the lab. The model is a question
answerer: English in, English out. Everything you read on screen is a language
you speak, which matters more than it sounds — the point of the exercise is to
compare what *you* understand about a sentence with what the model appears to be
doing, and you cannot do that in a language you are guessing at.

The model is `flan-t5-base`: 12 encoder layers, 12 decoder layers, 12 heads,
248M parameters. Bigger than the paper's base model at 65M, and still roughly
ten thousand times smaller than the models you use in a chat window.

It is an **encoder-decoder**, which is the architecture in Figure 1 of the paper,
and it is why we can look at all three kinds of attention in one sitting.

---
## 1. Load it

One cell, once per session. Colab forgets everything when the tab is closed, so
if you come back tomorrow, run this again. It downloads about 1 GB the first
time. You do not need an account, or an API key. There is nothing to pay for.

In [ ]:
# Colab has the torch, numpy, pandas and matplotlib libraries already installed.
# These are the libraries it does not.
%pip install --quiet "transformers>=4.40" sentencepiece bertviz

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# attn_implementation="eager" is not optional. The faster default ("sdpa") never
# builds the attention matrix we want to look at, and silently hands back an empty
# tuple instead of an error. Every cell below would fail with no explanation.
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, attn_implementation="eager")
model.eval()

cfg = model.config
print("encoder layers :", cfg.num_layers)
print("decoder layers :", cfg.num_decoder_layers)
print("attention heads:", cfg.num_heads)
print("d_model        :", cfg.d_model)
print("parameters     : %.0fM" % (sum(p.numel() for p in model.parameters()) / 1e6))

In [ ]:
def answer(prompt, max_new_tokens=20):
    """English in, English out."""
    enc = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new_tokens)
    return tokenizer.decode(out[0], skip_special_tokens=True)


for p in [
    "The trophy would not fit in the suitcase because it was too big. Question: What was too big?",
    "Answer in a full sentence. Passage: The library closes at nine on weekdays. Question: When does the library close?",
    "Question: What is the capital of Virginia?",
]:
    print(p)
    print("  ->", repr(answer(p)), "\n")

!!! tip "Your turn"

    **Edit the cell above.** Click into it, and change the three prompts in the
    list — the strings between the square brackets. Then run it again: **Shift and
    Enter together**, or the play button that appears at the left of the cell.

    Nothing here is read-only, and nothing you can type will break it. If a cell
    stops working, run the setup cell at the top again and carry on.

    Every section below this one asks you to change a value and re-run. This is
    that skill, on the easiest possible thing to change, so get it working now.

    Ask it something you can check. A fact about Hampden-Sydney, a question about a
    passage you paste in, a question with a name in it. **Find one it gets wrong.**

    It is a 248M-parameter model doing a job that GPT-4-class models do far better.
    Does the quality surprise you in either direction?

    Notice how much the phrasing matters. `Question:` in the prompt is not
    decoration — try removing it.

---
## 2. The three kinds of attention

Figure 1 of the paper has three attention blocks, and the model returns all three:

| | what attends | to what |
|:---|:---|:---|
| **encoder self-attention** | each input token | every input token |
| **decoder self-attention** | each answer token | answer tokens *already written* |
| **cross-attention** | each answer token | every input token |

One matrix per layer per head. To see attention over what the model *actually
said*, we let it answer first, then feed its own answer back in — otherwise you
are looking at attention for an answer it never gave.

In [ ]:
def look(prompt, max_new_tokens=20):
    """Answer, then re-run the model on its own answer to capture attention."""
    enc = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        generated = model.generate(**enc, max_new_tokens=max_new_tokens)
        out = model(
            input_ids=enc.input_ids,
            decoder_input_ids=generated,
            output_attentions=True,
        )
    source = tokenizer.convert_ids_to_tokens(enc.input_ids[0])
    written = tokenizer.convert_ids_to_tokens(generated[0])
    return source, written, out


def clean(tokens):
    """T5 marks a word start with the character U+2581. Drop it for display."""
    return [t.replace("\u2581", "") or "_" for t in tokens]


PROMPT = ("Answer in a full sentence. Passage: The library closes at nine on weekdays. "
          "Question: When does the library close?")
source, written, out = look(PROMPT)

print("input tokens :", clean(source))
print("answer tokens:", clean(written))
print()
print("encoder self-attention:", len(out.encoder_attentions), "layers,",
      "each", tuple(out.encoder_attentions[0].shape), "= (batch, heads, from, to)")
print("decoder self-attention:", len(out.decoder_attentions), "layers, each",
      tuple(out.decoder_attentions[0].shape))
print("cross-attention       :", len(out.cross_attentions), "layers, each",
      tuple(out.cross_attentions[0].shape))

### Two tokens that are not words

Look at the token lists above before going on. Two things in them will confuse you
later if you meet them for the first time inside a picture.

`</s>` is the **end-of-sentence token**. The tokenizer appends it to every input so
the model knows where the text stops. It carries no meaning. This model has no
matching `<s>` at the start — its only special tokens are `</s>`, `<unk>` and
`<pad>`.

`<pad>` is what the decoder is handed to start writing, before it has written
anything. Every answer begins with it.

And any token displaying as `_` is a **word-start marker for a word that got
split**. Rare words do not have their own entry in the vocabulary, so they arrive
in pieces. That is the Week 2 point about tokens not being words, turning up
somewhere it costs you something: the row you want to read may be split in two.

### The mask is a real thing

The paper says the decoder is masked so a position cannot attend to positions after
it — that is what stops the model reading the answer while writing it. You can check
the claim directly. Every number above the diagonal should be exactly zero.

In [ ]:
def heatmap(matrix, rows, cols, title, figsize=(7, 5.5)):
    """One attention matrix, drawn."""
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(matrix, cmap="viridis", vmin=0, aspect="auto")
    ax.set_xticks(range(len(cols)), clean(cols), rotation=90)
    ax.set_yticks(range(len(rows)), clean(rows))
    ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=0.8)
    fig.tight_layout()
    plt.show()


dec = out.decoder_attentions[0][0, 0].numpy()  # layer 0, head 0

above_diagonal = np.triu(dec, k=1)
print("largest weight above the diagonal:", above_diagonal.max())
print("each row sums to                 :", np.round(dec.sum(axis=1), 6))

heatmap(dec, written, written, "Decoder self-attention — layer 0, head 0")

!!! tip "Your turn"

    The triangle is the mask. Cover it with your hand and ask what the model would
    be able to do if it were not there — why would training break?

    Look along the first column. A lot of heads point hard at `<pad>`, the token
    the decoder starts from, which carries no meaning at all. What could a head be
    using it *for*? Hold that thought; section 3 answers it.

---
## 3. Predict before you look

Here is the exercise that matters. Take a sentence where a pronoun is genuinely
ambiguous:

> *The trophy would not fit in the suitcase because it was too big.*

You resolved `it` without noticing you did it. **Write down, now, which word you
think `it` will attend to.** Then run the cell.

Averaging over the heads of a layer is a blunt instrument — it is the first thing
anyone does and it hides as much as it shows. Section 4 takes the average apart.

In [ ]:
AMBIGUOUS = ("The trophy would not fit in the suitcase because it was too big. "
             "Question: What was too big?")
source, written, out = look(AMBIGUOUS)

print("the model answers:", repr(tokenizer.decode(
    tokenizer.convert_tokens_to_ids(written), skip_special_tokens=True)))
print()

LAYER = 3
enc_att = out.encoder_attentions[LAYER][0].mean(axis=0).numpy()  # average the 12 heads

heatmap(enc_att, source, source,
        f"Encoder self-attention — layer {LAYER}, averaged over {model.config.num_heads} heads")

WORD = "it"  # change this when you change the sentence
targets = [i for i, t in enumerate(source) if t.replace("\u2581", "") == WORD]
if not targets:
    raise SystemExit(f"{WORD!r} is not a token here. Tokens are: {clean(source)}")
row = targets[0]
weights = pd.Series(enc_att[row], index=clean(source)).sort_values(ascending=False)
print(f"What {WORD!r} attends to, most to least:")
print(weights.head(6).round(3))

### That is not what you predicted, and it looks broken. It is not.

**`</s>` takes the biggest weight, and it means nothing.**

Softmax forces every row to add to exactly 1. A head with nothing to contribute at
this position still has to put its weight *somewhere* — it has no way to abstain.
So it learns to park the weight where adding it back changes the vector least: on
a position that carries no meaning. `</s>` is the most convenient parking space in
the sentence, and `<pad>` is the decoder's. Heads doing this are sometimes called
**no-ops**, and the habit is common enough to have a name: an **attention sink**.

That is Tuesday's line — *a head with nothing to say still has to put its weight
somewhere* — as something you can see.

So reading these pictures is a skill with one trick at the front of it: **discount
the sinks first**, then look at what is left.

**Even discounting them, `it` does not point at a noun.**

Change `LAYER` above to any value from `0` to `11` and re-run that cell. There are
twelve encoder layers, which section 1 printed for you. In most of them the
leftover weight goes to `because`, or to `the`, and not to `trophy` or `suitcase`
— the thing you did, without effort, before you ran the cell.

Do not conclude the model failed. It answered the question correctly. Something in
there resolved the pronoun. Three things are hiding it:

- **You are looking at one layer of twelve, not at the model.** By layer 3 the
  vector at `it` has been rewritten three times, and whatever it collected in
  layers 0, 1 and 2 is inside it. Information can arrive in two hops — `it` reads
  `because`, `because` already read `trophy` — with no single arrow from `it` to
  `trophy` anywhere in the picture.
- **You are looking at twelve heads averaged together.** One head pointing hard at
  `trophy` while eleven park on `</s>` averages away to almost nothing. This is the
  big one, and section 4 is about it.
- **Attention is only the reading step.** Each block is attention *and then* a
  feed-forward layer working on each position separately, and nothing that layer
  does appears in an attention picture at all. Most of the parameters are there.

**Changing the layer, and looking at one head**

`LAYER = 3` is the only thing you need to touch above. To look at a single head
instead of the average of twelve, change the line under it:

```python
LAYER, HEAD = 3, 0
enc_att = out.encoder_attentions[LAYER][0, HEAD].numpy()   # one head
# enc_att = out.encoder_attentions[LAYER][0].mean(axis=0).numpy()   # all twelve, averaged
```

The indices are `[layer][batch, head]`. The batch is always `0` because you are
sending one sentence at a time. Heads run `0` to `11`.

---
## 4. Take the average apart

The average said `</s>`. Twelve heads went into that average. The obvious question
is whether any single one of them did the thing you did.

Instead of clicking through 144 pictures, score them. For every layer and every
head, ask one question: **when `it` looks, where does its largest weight land?**

In [ ]:
def where_it_looks(out, source, word="it"):
    """For every (layer, head), the token `word` attends to most strongly."""
    idx = [i for i, t in enumerate(source) if t.replace("\u2581", "") == word][0]
    rows = []
    n_layers = len(out.encoder_attentions)
    n_heads = out.encoder_attentions[0].shape[1]
    for layer in range(n_layers):
        att = out.encoder_attentions[layer][0].numpy()
        for head in range(n_heads):
            j = int(att[head][idx].argmax())
            rows.append({"layer": layer, "head": head,
                         "points at": clean(source)[j],
                         "weight": round(float(att[head][idx][j]), 3)})
    return pd.DataFrame(rows)


table = where_it_looks(out, source)
print("Where the", len(table), "heads send 'it':")
print(table["points at"].value_counts(), "\n")

nouns = table[table["points at"].isin(["trophy", "suitcase"])]
print("Heads whose 'it' row peaks on one of the two nouns:")
print(nouns.sort_values("weight", ascending=False).to_string(index=False))

!!! tip "Your turn"

    There it is. Several heads *do* put `it` on a noun, and the strongest of them is
    emphatic about it — far more confident than the averaged picture was about
    anything.

    Draw the best one with `heatmap` and look at the row. Use the single-head line
    from section 3.

    **Before you go on, decide what you now believe.** You predicted `trophy`. A
    head agrees with you, strongly. Is that head resolving the pronoun?

    Write your answer down. The next section tests it.

---
## 5. The control

There is a way to find out, and it is the whole of the scientific method in one
cell.

Change one word of the sentence so the answer flips. *Too big* means the trophy.
*Too small* means the suitcase — same sentence, same pronoun, opposite referent.
A head that is really resolving the pronoun must follow it. A head that only looks
like it is will point at the same word both times.

In [ ]:
PAIR = [
    ("The trophy would not fit in the suitcase because it was too big. "
     "Question: What was too big?", "trophy"),
    ("The trophy would not fit in the suitcase because it was too small. "
     "Question: What was too small?", "suitcase"),
]

# The heads that looked convincing in section 4. Change these to whatever you found.
CANDIDATES = [(10, 3), (8, 11), (7, 2), (4, 11)]

records = []
for prompt, expected in PAIR:
    src, wrt, o = look(prompt)
    said = tokenizer.decode(tokenizer.convert_tokens_to_ids(wrt), skip_special_tokens=True)
    idx = [i for i, t in enumerate(src) if t.replace("\u2581", "") == "it"][0]
    for layer, head in CANDIDATES:
        att = o.encoder_attentions[layer][0][head].numpy()[idx]
        j = int(att.argmax())
        records.append({"sentence": expected, "model answered": said,
                        "head": f"L{layer} H{head}",
                        "'it' points at": clean(src)[j],
                        "weight": round(float(att[j]), 2)})

df = pd.DataFrame(records)
print(df.to_string(index=False), "\n")

print("Does the head follow the referent, or always say the same word?")
for head, grp in df.groupby("head", sort=False):
    seen = list(grp["'it' points at"])
    verdict = "tracks it" if len(set(seen)) > 1 else f"always says {seen[0]!r} — not coreference"
    print(f"  {head}: {seen[0]} / {seen[1]}   ->  {verdict}")

### What just happened

The model answered **both** questions correctly. It said *the trophy* for the big
one and *the suitcase* for the small one. Whatever resolves that pronoun, it works.

And the head that convinced you in section 4 — the emphatic one, the one that
looked like a textbook coreference head — very likely pointed at `trophy` **both
times**, including in the sentence where the answer is the suitcase and the model
*said* the suitcase.

It was never resolving the pronoun. It was doing something much duller, like
pointing at the subject noun of the sentence, and on the first sentence that
happened to coincide with the right answer. You found a pattern, the pattern was
real, and your explanation of it was wrong.

This is the thing to take out of today, and it is not a fact about this model:

> An attention weight tells you where information was **read from**. It does not
> tell you what was **done with it** — and a picture that matches your explanation
> is not evidence that your explanation is right.

The way you tell the difference is what you just did: change one thing, hold the
rest fixed, and see whether the picture follows. A single heatmap can only ever
suggest a hypothesis. The control is what tests it.

Attention pictures get used in the wild to argue that a model is fair, or safe, or
attending to the clinically relevant part of a scan. Ask what the control was.

!!! tip "Your turn"

    Did any head track the referent across both sentences? If one did, it has
    earned exactly one more experiment, not your belief — write a third sentence
    and see if it survives that too.

    Then build your own pair. Two sentences differing in one word, where a human
    reader flips which noun a pronoun means. Run the same control.

    **A confusion you can state precisely is a result.** "I found a head that looked
    like coreference and it failed the control" is a finding, and it is most of the
    field's experience.

---
## 6. Cross-attention: where the answer is read from

The bridge. Every token the model writes can look at every token it read, at any
distance, all at once. This is the thing an RNN could not do.

It is tempting to read it as a **word alignment**: a table of which written word
came from which read word. You now know better than to trust an averaged picture,
so do what you did in section 4. Look at the average, expect sinks, then go hunting
for the head that is actually doing the work.

Keep this section in mind for Week 10. Retrieval-augmented generation is this
picture, scaled up: a model writing an answer while reading a passage it was handed.

In [ ]:
PASSAGE = ("Answer the question. Passage: Margaret sold her bicycle to Tom in April "
           "because she was moving to Chicago. Question: Why did Margaret sell the bicycle?")

src, wrt, o = look(PASSAGE)
print("answer:", repr(tokenizer.decode(
    tokenizer.convert_tokens_to_ids(wrt), skip_special_tokens=True)))

LAYER = 0
cross = o.cross_attentions[LAYER][0].mean(axis=0).numpy()
heatmap(cross, wrt, src,
        f"Cross-attention - layer {LAYER}, averaged over heads", figsize=(9, 5))

print("\nAveraged, for each token it wrote, the input token it looked at hardest:")
for i, tokn in enumerate(clean(wrt)):
    j = int(cross[i].argmax())
    print(f"  {tokn:12s} -> {clean(src)[j]:12s} {cross[i][j]:.2f}")

In [ ]:
# Same move as section 4: score every head instead of clicking through 144 pictures.
# A head is "an aligner" if the tokens it writes point at real words, not at sinks.
SINKS = {i for i, t in enumerate(clean(src)) if t in ("</s>", "<pad>")}

scored = []
for layer in range(len(o.cross_attentions)):
    att = o.cross_attentions[layer][0].numpy()
    for head in range(att.shape[0]):
        targets = [int(att[head][i].argmax()) for i in range(len(wrt))]
        content = sum(1 for t in targets if t not in SINKS)
        pairs = " ".join(
            clean(wrt)[i] + "->" + clean(src)[targets[i]] for i in range(1, len(wrt) - 1)
        )
        scored.append({"layer": layer, "head": head, "content": content, "alignment": pairs})

board = pd.DataFrame(scored).sort_values("content", ascending=False)
print("The heads that point at words rather than sinks:\n")
for _, r in board.head(4).iterrows():
    print("  layer %2d head %2d  (%d/%d)  %s"
          % (r["layer"], r["head"], r["content"], len(wrt), r["alignment"]))

!!! tip "Your turn"

    The average was almost all sinks, and it is at **every** layer — change `LAYER`
    and check a few. Yet somewhere in those 144 heads there is very likely one whose
    alignment reads straight down the diagonal, each written word pointing at the
    word it came from. That is a word alignment, and no averaged picture in this
    model shows it to you.

    Draw your winner and look at it:

    ```python
    LAYER, HEAD = 0, 0          # put your own winner here
    cross = o.cross_attentions[LAYER][0, HEAD].numpy()
    heatmap(cross, wrt, src, f"Cross-attention - layer {LAYER}, head {HEAD}", figsize=(9, 5))
    ```

    Some heads align *offset by one*, each written token pointing at the input word
    after the one it came from. What would a head like that be for?

    Then the question Week 10 turns on. If you handed this model a document and
    asked it a question, would this picture tell you whether it actually used the
    document, or only that some head read it?

---
## 7. If you would rather click than plot

`bertviz` renders the same numbers as an interactive picture — every layer and head,
all three attention types, in one widget. Use the dropdowns; the tabs across the top
switch between encoder, decoder and cross.

It is a browser for the same tensors you have been slicing by hand, not a different
source of truth. If it fails to render, nothing above depends on it.

In [ ]:
from bertviz import model_view

src, wrt, o = look(AMBIGUOUS)
model_view(
    encoder_attention=o.encoder_attentions,
    decoder_attention=o.decoder_attentions,
    cross_attention=o.cross_attentions,
    encoder_tokens=clean(src),
    decoder_tokens=clean(wrt),
)

---
## What to bring on Thursday

One finding. A claim plus the evidence you ran, in three sentences:

1. what you expected,
2. what the model actually did,
3. the layer, head and sentence, so somebody else can reproduce it.

A screenshot of a heatmap is fine. A table pasted into a document is fine. **A
confusion you can state precisely is a contribution** — "I found a head I cannot
explain, here it is" is a better ten minutes than a tidy result everybody nodded at.

Bring a laptop. If you have not opened this at all, come anyway and sit with someone
who has.